# Tokenization

In this notebook, we will build the first component needed to train a standard transformer language model from scratch, the tokenizer. We will train and implement a *byte-pair encoding* tokenizer. In particular, we will represent arbitrary unicode strings as a sequence of bytes and train our BPE tokenizer on this byte sequence. Later, we will use this tokenizer to encode a string of text into tokens, i.e. a sequence of integers, for language modeling.

**What we will do:**
1. Implement a byte-pair encoding (BPE) tokenizer.
2. Train a BPE tokenizer on the TinyStories dataset.
3. Run this trained tokenizer on the dataset to convert it into a sequence of integer IDs.

**Datasets:**
We will need the following two datasets in this notebook:
- TinyStories (~2.3 GB).
- OpenWebText Sample (~11.9 GB).

First, if needed, comment out the line below to pip install the required packages to your virtual environment.

In [1]:
# ! pip install regex line-profiler

In [2]:
import os
import re
import json
import csv
import time
import collections
from pathlib import Path

import regex
import numpy as np

import sys; sys.path.insert(0, '..')
from tests.test_tokenization import *
from tokenization_utils import *

%load_ext line_profiler

In [3]:
seed = 42
np.random.seed(seed)

In [4]:
tmp_dir = Path.cwd() / 'tmp'

## The Unicode Standard

*Unicode* is a text encoding standard that maps characters to integer "code points". As of Unicode 16.0, the standard defines 154,998 characters across 168 alphabetic scripts. For example, the character `s` has the code point `115`, typically notated as `U+0073`, where `U+` is a conventional prefix and `0073` is `115` in hexadecimal, usually written `0x0073`. The character `牛` has the code point `29275`, or `0x725b`, meaning its character is notated as `U+725b`.

In Python, you can use the `ord()` function to convert a single unicode character into its integer code point. The `chr()` function converts the code point back into its string representation with the corresponding character.

In [5]:
ord('牛')

29275

In [6]:
chr(29275)

'牛'

In [7]:
hex(29275)

'0x725b'

In [8]:
chr(0x725b)

'牛'

### Problem: Understanding Unicode

Run the code cells below to help answer the following questions:
1. What unicode character does `chr(0)` return?
2. How does this character's string representation `chr(0).__repr__()` differ from its printed `print(chr(0))` representation?
3. What happens when this character occurs in text? It may be helpful to play around with the following in your Python interpreter and see if it matches your expectations:

In [9]:
chr(0)

'\x00'

In [10]:
print(chr(0).__repr__())

'\x00'


In [11]:
print(chr(0))

 


In [12]:
s = f'this is a test{chr(0)}string'
s

'this is a test\x00string'

In [13]:
print(s)

this is a test string


## Unicode Encodings

While the unicode standard defines a mapping from characters to code points, it turns out to be impractical to train tokenizers directly on all unicode code points, since the vocabulary would be prohibitively large (around 150k items) and sparse (most characters are rare in real-world text). Thus, instead we more often use a unicode encoding, which converts each unicode character into a sequence of bytes. The unicode standard defines three such encodings: UTF-8, UTF-16, and UTF-32, with UTF-8 being the dominant encoding for the internet, accounting for more than 98% of all webpages.

To encode a unicode string into UTF-8 we can use the `encode()` function in Python. To access the underlying byte values for a Python `bytes` object we can iterate over it, e.g. with `list()`. Finally, we can use the `decode()` function to decode a UTF-8 byte string back into an ordinary unicode string.

Below we define an example unicode string in `string`. We then UTF-8 encode it using `encode()` to show its representation as a byte string `byte_string`. Notice that `string` has type `str`, while `byte_string` has type `bytes`. In Python, byte strings are typically prefixed with a `b''` to let you know it's a byte string.

In [14]:
string = 'hello! こんにちは!'
string, type(string)

('hello! こんにちは!', str)

In [15]:
byte_string = string.encode('utf-8')
byte_string, type(byte_string)

(b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!',
 bytes)

Given the byte string representation we can iterate byte by byte over the string and get each byte's integer reprentation. Since there are 8 bits in a byte, each byte corresponds to an integer between 0 and 255.

In [16]:
print(list(byte_string))

[104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]


Notice there isn't a 1-1 correspondance between the number of bytes in a string and the number of unicode characters in that string. In this example, `string` only has 13 unicode characters, but is represented by 23 bytes. This means some unicode characters are represented by multiple bytes.

In [17]:
len(string), len(byte_string)

(13, 23)

Nevertheless, we can always recover the original string simply by UTF-8 decoding the byte string back into unicode using the `decode()` method.

In [18]:
byte_string.decode('utf-8')

'hello! こんにちは!'

By encoding our unicode code points as byte sequences, we are essentially taking a sequence of code points, i.e. integers in the range `0-154994`, and transforming them into byte sequences where each byte lies in the smaller range `0-255`. This length 256 byte vocabulary is *much* more manageable to deal with. Moreover, when using byte-level tokenization we needn't to worry about out-of-vocabulary tokens, since we know that *any* input text can be expressed as a sequence of bytes.

#### Problem: Unicode Encodings

What are some reasons to prefer tokenizing on UTF-8 encoded bytes rather than, say, UTF-16 or UTF-32 encoded bytes? It might help to compare the output of these encodings for various input strings.

In [19]:
# answer
string = 'hello'
print(list(string.encode('utf-8')))
print(list(string.encode('utf-16')))
print(list(string.encode('utf-32')))

[104, 101, 108, 108, 111]
[255, 254, 104, 0, 101, 0, 108, 0, 108, 0, 111, 0]
[255, 254, 0, 0, 104, 0, 0, 0, 101, 0, 0, 0, 108, 0, 0, 0, 108, 0, 0, 0, 111, 0, 0, 0]


Consider the function `decode_bytes_to_str` below, which is intended to decode a UTF-8 byte string into a unicode string. Why is this implementation incorrect? What type of string might this implementation fail on?

In [20]:
def decode_bytes_to_str(byte_string):
    return ''.join([bytes([b]).decode('utf-8') for b in byte_string])

byte_string = 'hello'.encode('utf-8')
decode_bytes_to_str(byte_string)

'hello'

In [21]:
# answer
try:
    decode_bytes_to_str('hello! こんにちは!'.encode('utf-8'))
except UnicodeDecodeError as error:
    print(f'UnicodeDecodeError: {error}')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe3 in position 0: unexpected end of data


Not all bytes decode into unicode. Try to come up with an example of a two-byte sequence that doesn't decode to any unicode character(s).

In [22]:
# answer
b1, b2 = 255, 255
byte_string = b''.join([bytes([b1]), bytes([b2])])
try:
    byte_string.decode('utf-8')
except UnicodeDecodeError as error:
    print(f'UnicodeDecodeError: {error}')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte


## Subword Tokenization

While byte-level tokenization can alleviate the out-of-vocabulary issues faced by word-level tokenizers, tokenizing text into bytes results in extremely long input sequences. This slows down model training, since a sentence with 10 words might only be 10 tokens long in a word-level language model, but could be 50 or more tokens long in a character-level model (depending on the length of the words). Processing these longer sequences requires more computation at each step of the model. Furthermore, language modeling on byte sequences is difficult because the longer input sequences create long-term dependencies in the data. 

*Subword tokenization* is a happy midpoint between word-level tokenization and byte-level tokenization. A subword tokenizer trades off a *larger vocabulary size* for *better compression* of the input byte sequence. For example, if the byte sequence `b'the'` often occurs in our raw text training data, assigning it its own entry in the vocabulary would reduce this three-token sequence down to a single token.

How do we select which subword units to add to our vocabulary? *Sennrich et al. (2016)* propose to use *byte-pair encoding* or *BPE* (*Gage, 1994*), a compression algorithm that iteratively replaces or "merges" the most frequent pair of bytes with a single, new unused index. In essense, this algorithm adds extra subword tokens to our vocabulary in order to maximize the compression of our input sequences. If a subword occurs enough times it'll be represented as a single subword unit.

Subword tokenizers with vocabularies constructed via BPE are often called *BPE tokenizers*. In the next problem we'll implement a byte-level BPE tokenizer, where the vocabulary items are bytes or merged sequences of bytes, which give us the best of both worlds in terms of out-of-vocabulary handling and manageable input sequence lengths. The process of constructing a BPE tokenizer vocabulary on a given corpus is often referred to as "training" the BPE tokenizer even though there are no parameters being created or updated here.

## BPE Tokenizer Training

The BPE tokenizer training procedure consists of three main steps.

**Vocabulary initialization:** The tokenizer vocabulary is a 1-1 mapping from byte strings to integer IDs. For a byte-level BPE tokenizer we simply set the initial vocabulary to all bytes, resulting in an initial vocabulary size of 256.

**Pre-tokenization:** Once we've got an initial vocabulary we could in principle then count how often bytes occur next to each other in the text and begin merging them starting with the most frequent pair of bytes. However, this is quite computationally expensive since we'd have to do a full pass over the corpus each time we need to merge. 

Another issue is that if we directly merge bytes across the corpus like this, it may result in tokens that differ only in punctuation, e.g. `dog!` vs. `dog.` would be different tokens. These tokens would get completely different token IDs even though they're likely to have high semantic similarity, since they only differ in punctuation.

To avoid this, we can *pre-tokenize* the corpus. You can think of pre-tokenization as a kind of coarse-grained tokenization over the corpus to help us count how often pairs of characters appear. For example, the word `text` might appear as a pre-token 10 times. In this case, when we count how often the characters `t` and `e` appear next to each other, we will see that the word `text` has `t` and `e` adjacent and we can increment their count by 10 instead of looking through the entire corpus.

The original BPE implementation of *Sennrich et al.* pre-tokenizes by simply splitting on whitespace, e.g. `s.split(' ')`. In contrast, we'll use a smarter regex-based pre-tokenizer, in particular the GPT-2 pre-tokenizer (*Radford et al., 2019*) regex string `gpt2_pretokenizer_pattern` defined below, taken as-is from the [tiktoken](https://github.com/openai/tiktoken/pull/234/files) repository. 

```python
gpt2_pretokenizer_pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
```

It may be useful to interactively split some text with this pre-tokenizer to get a better sense of its behavior. *Note:* For effiency reasons, when using this pre-tokenizer in your code, consider using `regex.finditer` to avoid storing the pre-tokenized words as you construct your mapping from pre-tokens to their counts.

In [23]:
gpt2_pretokenizer_pattern

"'(?:[sdmt]|ll|ve|re)| ?\\p{L}+| ?\\p{N}+| ?[^\\s\\p{L}\\p{N}]+|\\s+(?!\\S)|\\s+"

In [24]:
regex.findall(gpt2_pretokenizer_pattern, "some text that I'll pre-tokenize")

['some', ' text', ' that', ' I', "'ll", ' pre', '-', 'tokenize']

In [25]:
pattern_iter = regex.finditer(gpt2_pretokenizer_pattern, "some text that I'll pre-tokenize")
for text in pattern_iter:
    print(text)

<regex.Match object; span=(0, 4), match='some'>
<regex.Match object; span=(4, 9), match=' text'>
<regex.Match object; span=(9, 14), match=' that'>
<regex.Match object; span=(14, 16), match=' I'>
<regex.Match object; span=(16, 19), match="'ll">
<regex.Match object; span=(19, 23), match=' pre'>
<regex.Match object; span=(23, 24), match='-'>
<regex.Match object; span=(24, 32), match='tokenize'>


**Compute BPE merges:** Now that we've converted our input text into pre-tokens and represented each pre-token as a sequence of UTF-8 bytes, we can compute the BPE merges, i.e. "train" the BPE tokenizer. At a high level, the BPE algorithm iteratively counts every pair of bytes and identifies the pair with the highest frequency (`A`, `B`). Every occurrence of this most frequent pair (`A`, `B`) is then merged, i.e. replaced with a new merged token `AB`. This new token is then added to our vocabulary. As a result, the final vocabulary after BPE training is the size of the initial vocabulary plus the number of BPE merge operations performed during training. 

For efficiency, during training we don't consider pairs that cross pre-token boundaries. Also, when computing merges we deterministically break ties in pair frequency by choosing the lexicographically greater pair. For example, if the pairs (`A`, `B`), (`A`, `C`), (`B`, `ZZ`), (`BA`, `A`) all have the highest frequency, we'd merge (`BA`, `A`).

In [26]:
max([('A', 'B'), ('A', 'C'), ('B', 'ZZ'), ('BA', 'A')])

('BA', 'A')

*Note:* The original BPE formulation also used an *end-of-word token*, denoted `</w>`. Nowadays, we never add an end-of-word token when training byte-level BPE models since all bytes, including whitespace and punctuation, are included in the model's vocabulary. Since we already explicitly representing spaces and punctuation the learned BPE merges will naturally reflect these word boundaries anyway.

**Special tokens:** Some special strings like `<|endoftext|>` are used to encode metadata, e.g. boundaries between documents. When encoding text it's often desirable to treat these kinds of strings as "special tokens" that should never be split into multiple tokens, hence will always be preserved as a single token. For example, the end-of-sequence string `<|endoftext|>` should always be preserved as a single token so we know when to stop generating from the language model. These special tokens must be manually added to the vocabulary so that they have a corresponding fixed token ID.

*Sennrich et al.* provide an implementation of BPE tokenizer training in their *Algorithm 1* example, shown below. Their implementation essentially follows the steps outlined above. It turns out, however, that it's inefficient as is. Indeed, it may be helpful to look closely at their implementation and identify why it's inefficient.

In [27]:
def get_stats(vocab):
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i + 1]] += freq
    return pairs

def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

vocab = {'l o w </w>' : 5, 'l o w e r </w>' : 2, 'n e w e s t </w>': 6, 'w i d e s t </w>': 3}
num_merges = 10
for i in range(num_merges):
    pairs = get_stats(vocab)
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(best)

('e', 's')
('es', 't')
('est', '</w>')
('l', 'o')
('lo', 'w')
('n', 'e')
('ne', 'w')
('new', 'est</w>')
('low', '</w>')
('w', 'i')


#### Example: BPE training example

Here's another stylized example from *Sennrich et al*. Consider a corpus containing the following text:

```
low low low low low
lower lower widest widest widest
newest newest newest newest newest newest
```

*Vocabulary:* We initialize our vocabulary with our special token `<|endoftext|>` and the 256 byte values.

*Pre-tokenization:* For simplicity and to focus on the merge procedure, we'll assume in this example that pre-tokenization only splits on whitespace. When we pre-tokenize and count, we end up with the following frequency table:

```
{low: 5, lower: 2, widest: 3, newest: 6}
```

Note that it'll be more convenient to represent frequency tables in the following equivalent but slightly different form:

```
{(l,o,w): 5, (l,o,w,e,r): 2, (w,i,d,e,s,t): 3, (n,e,w,e,s,t): 6}
```

*Merges:* We first look at every successive pair of bytes and sum the frequency of the words where they appear:

```
{lo: 7, ow: 7, we: 8, er: 2, wi: 3, id: 3, de: 3, es: 9, st: 9, ne: 6, ew: 6}
``` 

The pair `es` and `st` have tied counts, so we'll choose the lexicographically greater pair, `st`. We'd then merge these pre-tokens so that we end up with:

```
{(l,o,w): 5, (l,o,w,e,r): 2, (w,i,d,e,st): 3, (n,e,w,e,st): 6}
```

In the second round, we find `(e, st)` is the most common pair with a count of 9, and so we'd then merge into:

```
{(l,o,w): 5, (l,o,w,e,r): 2, (w,i,d,est): 3, (n,e,w,est): 6}
```

Continuing on, the sequence of merges we end up doing will be:

```
['s t', 'e st', 'o w', 'l ow', 'w est', 'n e', 'ne west', 'w i', 'wi d', 'wid est', 'low e', 'lowe r']
```

If we take 6 merges, we have:

```
['s t', 'e st', 'o w', 'l ow', 'w est', 'n e']
```

And our vocabulary elements would be:

```
[<|endoftext|>, [0...255 BYTE CHARS], st, est, ow, low, west, ne]
```

Once we have this final vocabulary and set of merges, the word `newest` would then tokenize as `[ne, west]`.

## Experimenting with BPE Tokenizer Training

We'll now train a byte-level BPE tokenizer on the [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) dataset. We'll use the HuggingFace Datasets version of TinyStories which consists of two files, the training dataset `TinyStoriesV2-GPT4-train.txt`, and the validation dataset `TinyStoriesV2-GPT4-valid.txt`.

If you're using a UNIX system, you can just download these datasets to the local `tmp/` directory using the shell commands provided below. Note the two datasets will take several minutes to download and consume about 2.3 GB of disk. Be sure to comment out this cell again once the download is finished so the kernel doesn't re-download every time you run this notebook.

In [28]:
# %%bash

# if [ ! -d tmp ]; then mkdir tmp; fi
# cd ./tmp
# wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt >> /dev/null 2>&1
# wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt >> /dev/null 2>&1
# cd ..
# echo "Done."

Before we continue it's worth taking a look at the TinyStories dataset to get a sense of what it looks like. Each file is a simple `txt` containing the entire corpus in plain text. Below we'll load and inspect the first few lines of the training corpus. Feel free to modify this code to view other parts of this file, or the validation corpus.

*Note:* These files are quite large, so you don't want to read them into memory all at once. Notice below we open the file, stream and print only the first 100 lines, and then break. If we tried to read the whole file and then print it would take forever or the kernel might crash.

In [29]:
ts_train_path = tmp_dir / 'TinyStoriesV2-GPT4-train.txt'
ts_valid_path = tmp_dir / 'TinyStoriesV2-GPT4-valid.txt'

In [30]:
! cat {ts_train_path} | head -n 25


Once upon a time there was a little boy named Ben. Ben loved to explore the world around him. He saw many amazing things, like beautiful vases that were on display in a store. One day, Ben was walking through the store when he came across a very special vase. When Ben saw it he was amazed!  
He said, “Wow, that is a really amazing vase! Can I buy it?” 
The shopkeeper smiled and said, “Of course you can. You can take it home and show all your friends how amazing it is!”
So Ben took the vase home and he was so proud of it! He called his friends over and showed them the amazing vase. All his friends thought the vase was beautiful and couldn't believe how lucky Ben was. 
And that's how Ben found an amazing vase in the store!
<|endoftext|>
Once upon a time, there was a reliable otter named Ollie. He lived in a river with his family. They all loved to play and swim together.
One day, Ollie's mom said, "Ollie, hurry and get some fish for dinner!" Ollie swam fast to catch fish. He saw his fri

Note the use of the `<|endoftext|>` to delimit each document in the text corpus. We can use this token to split the corpus up into documents and figure out how many documents are in each dataset. Evidently there are 2.7M documents in the training dataset, and only 27.6k documents in the validation set.

In [31]:
def count_docs(text_path, end_doc_token='<|endoftext|>'):
    n_docs = 0
    with open(text_path, 'r') as file:
        for line in file:
            n_docs += line.count(end_doc_token)
    return n_docs

In [32]:
# count_docs(ts_train_path) # 2717699

In [33]:
count_docs(ts_valid_path) # 27630

27630

**Parallelizing pre-tokenization:** We'll often find that a major bottleneck in BPE training is the pre-tokenization step. We can speed up the pre-tokenization by parallelizing the code, e.g. using the `multiprocessing` library. When pre-tokenizing in parallel it's recommended to chunk the corpus while ensuring the chunk boundaries occur at the beginning of a special token. For a reference on how to obtain the chunk boundaries in parallel, see the example code in `refs/pretokenization_example.py`.

This chunking will always be valid since we never want to merge across document boundaries. For the purposes of this notebook we'll always split in this way. There is one edge case though, which is when we have a very large corpus without a `<|endoftext|>` token, though we'll ignore it here.

**Removing special tokens before pre-tokenization:** Before running pre-tokenization with the regex pattern (and `re.finditer`), it's good to first strip out all special tokens from the corpus (or chunk if using a parallel implementation). This can be achieved by simply splitting on the special tokens so that no merging occurs across the text they delimit. For example, for a corpus like `[Doc 1]<|endoftext|>[Doc 2]`, we'd split on the `<|endoftext|>` token and pretokenize `[Doc 1]` and `[Doc 2]` separately. That way, no merging occurs across document boundaries. This splitting can be done using `re.split` with `'|'.join(special_tokens)` as the delimiter, with careful use of `re.escape` since `|` may occur in the special tokens.

**Optimizing the merging step:** The naive implementation of BPE training from before is slow, because for every merge it iterates over all byte pairs to identify the most frequent pair. However, the only pair counts that change after each merge are those that overlap with the merged pair. Thus, BPE training speed can be improved by indexing the counts of all pairs and incrementally updating these
counts rather than explicitly iterating over each pair of bytes to count pair frequencies. We can get significant speedups with this caching trick, even though the merging part itself of BPE training is not parallelizable in Python.

*Tip:* Consider using profiling tools to identify the bottlenecks in your implementation and focus on optimizing those.

*Tip:* Instead of jumping to training your tokenizer on the full TinyStories dataset, it's a good idea to first train on a small subset of the data, a "debug dataset". For example, you could train your tokenizer on the TinyStories validation set instead, which is only 22K documents instead of 2.1M. 

This downscaling trick is a commonly employed strategy across all of machine learning, where we might use smaller datasets or smaller model sizes to test an implementation first before scaling up to full size. Choosing the size of the debug dataset or hyperparameter config requires careful consideration. You want your debug set to be large enough to have the same bottlenecks as the full configuration (so that the optimizations you make will generalize), but not so big that it takes forever to run.

#### Problem: BPE Tokenizer Training

Fill in the `train_bpe` function below. This function should take in a given path to an input text file, train a *byte-level* BPE tokenizer, and return the vocabulary dictionary along with the list of merge pairs.

To test your BPE training function we will use the `run_train_bpe` adapter filled in below. Try to make sure all tests are passing before proceeding to the next section.

In [34]:
def train_bpe(input_path, vocab_size, special_tokens, pretokenizer_pattern=gpt2_pretokenizer_pattern):
    """
    Given the path to an input corpus, train a BPE tokenizer and output its vocabulary and merges.
    Args:
        input_path: (str | os.PathLike). Path to BPE tokenizer training data.
        vocab_size: int. Total number of items in the tokenizer's vocabulary (including special tokens).
        special_tokens: list[str]. A list of string special tokens to be added to the tokenizer vocabulary.
            These strings will never be split into multiple tokens, and will always be kept as a single token. 
            If these special tokens occur in the input_path, they are treated as any other string.
        pretokenizer_pattern: str. Pretokenization pattern. Defaults to GPT-2 pretokenizer pattern.
    Returns:
        tuple[dict[int, bytes], list[tuple[bytes, bytes]]]
            vocab: dict[int, bytes].
                The trained tokenizer vocabulary, a mapping from int (token ID in the vocabulary) to bytes (token bytes)
            merges: list[tuple[bytes, bytes]].
                BPE merges. Each list item is a tuple of bytes (<token1>, <token2>), representing that <token1> was merged with <token2>.
                Merges are ordered by order of creation.
    """
    vocab = {idx: bytes([idx]) for idx in range(256)}
    for i, special_token in enumerate(special_tokens):
        vocab[256 + i] = special_token.encode('utf-8')
    pretoken_freqs = pretokenize(input_path, special_tokens, pretokenizer_pattern)
    pretoken_seqs, pretoken_freqs_by_id, pair_counts, pair_index = get_pair_counts_and_index(pretoken_freqs)
    n_merges = vocab_size - len(vocab)
    merges = []
    heap = []
    for pair, count in pair_counts.items():
        if count > 0:
            heap_push(heap, pair, count, vocab)
    for _ in range(n_merges):
        best_pair = heap_pop(heap, pair_counts)
        if best_pair is None or pair_counts.get(best_pair, 0) <= 0:
            break
        a, b = best_pair
        new_id = len(vocab)
        vocab[new_id] = vocab[a] + vocab[b]
        modified = merge_pair(pretoken_seqs, pretoken_freqs_by_id, pair_counts, best_pair, new_id, pair_index)
        merges.append((vocab[a], vocab[b]))
        for p in modified:
            count = pair_counts.get(p, 0)
            if count > 0:
                heap_push(heap, p, count, vocab)
    return vocab, merges

In [35]:
def run_train_bpe(input_path, vocab_size, special_tokens):
    """Adapter function for the main train_bpe function."""
    return train_bpe(input_path, vocab_size, special_tokens)

test_train_bpe(run_train_bpe)
test_train_bpe_special_tokens(run_train_bpe)
test_train_bpe_speed(run_train_bpe)

PASSED: test_train_bpe
PASSED: test_train_bpe_special_tokens
PASSED: test_train_bpe_speed


### Problem: BPE Training on TinyStories

Let's now train our BPE tokenizer on the TinyStories dataset. Use a maximum vocabulary size of 10,000, and don't forget to add the TinyStories `<|endoftext|>` special token to the vocabulary. As a goal, try to have this run in under 30 minutes on a CPU and use less than 30GB of RAM. You may wish to serialize the resulting vocabulary and merges to disk for future use. 

*Questions:* How many hours and how much memory did training take? What is the longest token in the vocabulary? Does it make sense?

*Hint:* You should be able to get under 2 minutes for BPE training by using `multiprocessing` during pre-tokenization, plus the fact that the `<|endoftext|>` token delimits documents in the file and is handled as a special case before BPE merges are applied. To do this, it helps to first profile your function. 

To profile your code, you can run a line profiler in a notebook using the `%lprun` magic, which relies on the Python `line-profiler` package being installed and `%load_ext line_profiler` being loaded first. For example, to profile `train_bpe` you might do something like this, replacing the function arguments with your own of course:

```python
%lprun -f train_bpe train_bpe(input_path=input_path, vocab_size=vocab_size, special_tokens=special_tokens)
```

In [36]:
# %lprun -f train_bpe train_bpe(input_path=ts_valid_path, vocab_size=10_000, special_tokens=['<|endoftext|>'])

In [37]:
start = time.time()
ts_vocab, ts_merges = train_bpe(input_path=ts_train_path, vocab_size=10_000, special_tokens=['<|endoftext|>'])
end = time.time()
print(f'Time: {round((end - start) // 60)} min {round((end - start) % 60)} sec') # 1 min 7 sec

Time: 1 min 7 sec


In [38]:
len(ts_vocab), len(ts_merges) # (10000, 9743)

(10000, 9743)

In [39]:
list(ts_vocab.items())[-15:] # [(9985, b'Froggy'), (9986, b' wrapper'), (9987, b' Reddy'), (9988, b' Hops'), (9989, b' Crusty'), ...]

[(9985, b'Froggy'),
 (9986, b' wrapper'),
 (9987, b' Reddy'),
 (9988, b' Hops'),
 (9989, b' Crusty'),
 (9990, b' whiskers'),
 (9991, b' nicest'),
 (9992, b' improving'),
 (9993, b' booth'),
 (9994, b' Land'),
 (9995, b'Surrender'),
 (9996, b'Rocky'),
 (9997, b' meadows'),
 (9998, b' imaginary'),
 (9999, b' bold')]

In [40]:
max_len = 0
for id in ts_vocab:
    if len(ts_vocab[id]) > max_len:
        max_token = ts_vocab[id]
        max_len = len(ts_vocab[id])
max_len, max_token.decode('utf-8') # (15, ' accomplishment')

(15, ' accomplishment')

In [41]:
ts_merges[:10] # [(b' ', b't'), (b'h', b'e'), (b' ', b'a'), (b' ', b's'), ...]

[(b' ', b't'),
 (b'h', b'e'),
 (b' ', b'a'),
 (b' ', b's'),
 (b' ', b'w'),
 (b'n', b'd'),
 (b' t', b'he'),
 (b'e', b'd'),
 (b' ', b'b'),
 (b' t', b'o')]

We'll now go ahead and save the `vocab` and `merges` objects to `tmp/` so we can easily reuse them later, without having to rerun BPE training every single time. To do this we'll use the `save_bpe_train_output` function below, which saves the `vocab` as a JSON file and the `merges` as a CSV file, with each byte object hex-encoded to ensure correctness when reading these files in later.

In [42]:
def save_bpe_train_output(vocab, merges, vocab_path, merges_path):
    """Saves vocab to json file vocab_path and merges to csv file merges_path"""
    vocab = {str(id): list(b) for id, b in vocab.items()}
    with open(vocab_path, 'w') as file:
        json.dump(vocab, file)
    with open(merges_path, 'w', newline='') as file:
        csv_writer = csv.writer(file)
        for a, b in merges:
            csv_writer.writerow([a.hex(), b.hex()])

In [43]:
ts_vocab_path = tmp_dir / 'ts-vocab.json'
ts_merges_path = tmp_dir / 'ts-merges.csv'
save_bpe_train_output(ts_vocab, ts_merges, ts_vocab_path, ts_merges_path)

---

Next, we'll try training a byte-level BPE tokenizer on the larger [OpenWebText](https://huggingface.co/datasets/stanford-cs336/owt-sample) dataset. We'll again use the HuggingFace Datasets version of this dataset, which like TinyStories is also split into separate training and validation datasets. This time the datasets are zipped as `gz` files. Once unzipped they should consume about 7.75 GB of disk.

UNIX users can download these datasets to the local `tmp/` directory by running the shell commands below. Expect this to take at least 10-20 minutes to complete, possibly longer. Again, be sure to comment out these lines once downloading is finished.

In [44]:
# %%bash

# if [ ! -d tmp ]; then mkdir tmp; fi
# cd ./tmp
# wget https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_train.txt.gz >> /dev/null 2>&1
# wget https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_valid.txt.gz >> /dev/null 2>&1
# gunzip owt_train.txt.gz
# gunzip owt_valid.txt.gz
# cd ..
# echo "Done."

Let's see how large these datasets are and what their contents look like.

In [45]:
owt_train_path = tmp_dir / 'owt_train.txt'
owt_valid_path = tmp_dir / 'owt_valid.txt'

In [46]:
# count_docs(owt_train_path) # 2399397

In [47]:
count_docs(owt_valid_path) # 59059

59059

In [48]:
# print first document in owt_train.txt up to first <|endoftext|> token
! awk '{{print}} /<\|endoftext\|>/{{exit}}' "{owt_train_path}"

What wouldn't you do to save someone you love?

When They Come Calling is a modern ghost story, a suspenseful weaving of urban battles, romance, and supernatural intrigue.

Anna is a physician from Kansas City. She’s spent her life giving and caring to others while trying to hide how different she is. Anna has lost everyone she’s ever loved: her relationships, her family, and her hope for something more.

Jed is a warrior from another era, haunted by the horrors of a brutal family feud three thousand years in the making, and inspired by his own secret quest. Relentless and driven, Jed’s determination radiates to everyone around him.

What happens when they meet will keep you turning the pages.

Author Sarah Fleming Mountford started with a short ghost story. Over years and through personal trials, she added to it and shaped it, until it was the story she always wanted to write. We are excited to present that story to you, and your pre-order pledges will help cover the costs to edit and

### Problem: BPE Training on OpenWebText

Train a byte-level BPE tokenizer on the OpenWebText dataset. This time use a maximum vocabulary size of 32,000. As a goal, try to have this run in under 12 hours on the CPU and use less than 100 GB of RAM. Again, you may want to serialize the vocabulary and merges to disk for later use. What is the longest token in the vocabulary? Does it make sense? Compare and contrast the tokenizer that you get training on TinyStories versus OpenWebText.

In [49]:
start = time.time()
owt_vocab, owt_merges = train_bpe(input_path=owt_train_path, vocab_size=32_000, special_tokens=['<|endoftext|>'])
end = time.time()
print(f'Time: {round((end - start) // 60)} min {round((end - start) % 60)} sec') # 20 min 5 sec

Time: 19 min 11 sec


In [50]:
len(owt_vocab), len(owt_merges) # (32000, 31743)

(32000, 31743)

In [51]:
list(owt_vocab.items())[-15:] # [(31985, b' Hz'), (31986, b' Aristotle'), (31987, b'ulls'), (31988, b'ircraft'), (31989, b'YD'), ...]

[(31985, b' Hz'),
 (31986, b' Aristotle'),
 (31987, b'ulls'),
 (31988, b'ircraft'),
 (31989, b'YD'),
 (31990, b'Wik'),
 (31991, b' partnering'),
 (31992, b' drilled'),
 (31993, b'hang'),
 (31994, b' pesticide'),
 (31995, b' latitude'),
 (31996, b' fetish'),
 (31997, b' Fruit'),
 (31998, b'ivists'),
 (31999, b'Disp')]

In [52]:
max_len = 0
for id in owt_vocab:
    if len(owt_vocab[id]) > max_len:
        max_token = owt_vocab[id]
        max_len = len(owt_vocab[id])
max_len, max_token.decode('utf-8') # (64, 'ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ')

(64, 'ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ')

In [53]:
owt_merges[:10] # [(b' ', b't'), (b' ', b'a'), (b'h', b'e'), (b'i', b'n'), (b'r', b'e'), ...]

[(b' ', b't'),
 (b' ', b'a'),
 (b'h', b'e'),
 (b'i', b'n'),
 (b'r', b'e'),
 (b' t', b'he'),
 (b'o', b'n'),
 (b'e', b'r'),
 (b' ', b's'),
 (b' ', b'w')]

In [54]:
owt_vocab_path = tmp_dir / 'owt-vocab.json'
owt_merges_path = tmp_dir / 'owt-merges.csv'
save_bpe_train_output(owt_vocab, owt_merges, owt_vocab_path, owt_merges_path)

## BPE Tokenizer: Encoding and Decoding

We've now implemented a function to train a BPE tokenizer on input text to obtain a tokenizer vocabulary and a list of BPE merges. Now, we will implement a BPE tokenizer that loads a provided vocabulary and list of merges and uses them to encode and decode text to or from token IDs.

### Encoding text

The process of encoding text by BPE mirrors how we train the BPE vocabulary. The major steps are:
1. *Pre-tokenize:* First pre-tokenize the sequence and represent each pre-token as a sequence of UTF-8 bytes, just as we did before. We'll want to merge the bytes within each pre-token into vocabulary elements, handling each pre-token independently with no merges across pre-token boundaries.
2. *Apply the merges:* We'll then take the sequence of vocabulary element merges created during BPE training and apply it to our pre-tokens in the *same order* of creation.

**Example: BPE encoding example**

Suppose our input string is `'the cat ate'`. The vocabulary is then:

```
{0: b' ', 1: b'a', 2: b'c', 3: b'e', 4: b'h', 5: b't', 6: b'th', 7: b' c', 8: b' a', 9: b'the', 10: b'at'}
```

The learned merges are:

```
[(b't', b'h'), (b' ', b'c'), (b' ', 'a'), (b'th', b'e'), (b' a', b't')]
```

First, the pre-tokenizer would split this string into:

```
['the', ' cat', ' ate']
```


Then we look at each pre-token and apply the BPE merges. The first pre-token `'the'` is initially represented as:

```
[b't', b'h', b'e']
```

Looking at the list of merges, the first applicable merge appears to be `(b't', b'h')`, which we can use to transform the pre-token into:

```
[b'th', b'e']
```

The next applicable merge appears to be `(b'th', b'e')`, which transforms the pre-token into `[b'the']`. 

Finally, looking back at the list of merges, there are no more that apply to the string since the entire pre-token has been merged into a single token, so we are done applying the BPE merges. The corresponding integer sequence is `[9]`.

Repeating this process for the remaining pre-tokens, we see that the pre-token `' cat'` is represented as follows after applying BPE merges:

```
`[b' c', b'a', b't']`
```

This then becomes the integer sequence `[7, 1, 5]`. 

The final pre-token `' ate'` after applying BPE merges is:

```
[b' at', b'e']
```

This becomes the integer sequence `[10, 3]`. Thus, the final result of encoding our input string is `[9, 7, 1, 5, 10, 3]`.

*Special tokens:* Our tokenizer should be able to properly handle user-defined special tokens when encoding text, provided when constructing the tokenizer.

*Memory considerations:* Suppose we want to tokenize a large text file that won't fit in memory. To efficiently tokenize this large file (or any other stream of data), we need to break it up into manageable chunks and process each chunk in turn so the memory complexity stays constant instead of being linear in the size of the text. In doing so, we need to make sure that a token doesn't cross chunk boundaries, else we'll get a different tokenization than the naive method of tokenizing the entire sequence in memory.

### Decoding text

To decode a sequence of integer token IDs back to raw text, we can simply look up each ID's corresponding entries in the vocabulary byte sequence, concatenate them together, and then decode the bytes to a unicode string. Note that input IDs are not guaranteed to map to valid unicode strings since a user could input any sequence of integer IDs. In the case that the input token IDs do not produce a valid
unicode string, we'd replace the malformed bytes with the official unicode replacement character `U+FFFD`. The errors argument of `bytes.decode` controls how unicode decoding errors are handled, and using `errors='replace'` will automatically replace malformed data with the replacement marker.

### Problem: Implementing the tokenizer

Let's now implement a `Tokenizer` class that, given a vocabulary and a list of merges, encodes text into integer IDs and decodes integer IDs into text. This tokenizer should also support user-provided special tokens, appending them to the vocabulary if they aren't already there. Follow the interface shown below when implementing this class.

To test your Tokenizer against our provided tests, we will use the `Tokenizer` class you defined to implement the test adapter function `get_tokenizer`. This adapter will then be used to execute a series of tests of your `Tokenizer` class. Try to make sure all tests are passing before proceeding.

In [55]:
class Tokenizer():
    def __init__(self, vocab, merges, special_tokens=None, pretokenizer_pattern=gpt2_pretokenizer_pattern):
        """
        Construct a tokenizer from a given vocabulary, list of merges, and (optionally) a list of special tokens. 
        Args:
            vocab: dict[int, bytes]. Dictionary mapping of token IDs to bytes.
            merges: list[tuple[bytes, bytes]]. List of merge pairs.
            special_tokens: list[str] | None = None. Special tokens to add to vocab, e.g. <|endoftext|>.
            pretokenizer_pattern: str. Pretokenization pattern. Defaults to GPT-2 pretokenizer pattern.
        """
        self.vocab = vocab
        self.merges = merges
        self.special_tokens = special_tokens if special_tokens else []
        self.pretokenizer_pattern = pretokenizer_pattern
        self.reverse_vocab = {byte: id for id, byte in vocab.items()}
        self.merge_ranks = {merge_pair: idx for idx, merge_pair in enumerate(merges)}
        # insert any special_tokens not already in vocab into vocab
        for special_token in self.special_tokens:
            vocab_size = len(self.vocab)
            special_token_bytes = special_token.encode('utf-8')
            if special_token_bytes not in self.reverse_vocab:
                self.vocab[vocab_size] = special_token_bytes
                self.reverse_vocab[special_token_bytes] = vocab_size
    
    @classmethod
    def from_files(cls, vocab_path, merges_path, special_tokens=None):
        """
        Class method that constructs and returns a Tokenizer from a serialized vocabulary and list of merges 
        (in the same format as the BPE training code output above) and optionally a list of special tokens.
        Args:
            vocab_path: (str | os.PathLike). Path to vocab dictionary. Assumes vocab is saved in a json file.
            merges_path: (str | os.PathLike). Path to merge pairs. Assumes merge pairs are saved in a csv file.
            special_tokens: list[str] | None = None. Special tokens to add to vocab.
        Returns:
            :object:Tokenizer. Initialized Tokenizer class object.
        """
        with open(vocab_path, 'r') as file:
            vocab = json.load(file)
            vocab = {int(id): bytes(b) for id, b in vocab.items()}
        with open(merges_path, mode='r', newline='', encoding='utf-8') as file:
            csv_reader = csv.reader(file)
            merges = [(bytes.fromhex(row[0]), bytes.fromhex(row[1])) for row in csv_reader]
        return Tokenizer(vocab, merges, special_tokens=special_tokens)
    
    def encode(self, text):
        """
        Encode an input text into a sequence of token IDs.
        Args:
            text: str. String to encode as token IDs.
        Returns:
            ids: list[int]. Token IDs representing encoded string.
        """
        sorted_tokens = sorted(self.special_tokens, key=len, reverse=True)
        split_pattern = '(' + '|'.join(re.escape(special_token) for special_token in sorted_tokens) + ')'
        chunks = re.split(split_pattern, text) if self.special_tokens else [text]
        special_token_set = set(self.special_tokens)
        ids = []
        for chunk in chunks:
            if not chunk:
                continue
            if chunk in special_token_set:
                ids.append(self.reverse_vocab[chunk.encode('utf-8')])
                continue
            for match in regex.finditer(self.pretokenizer_pattern, chunk):
                ids.extend(self.reverse_vocab[tok] for tok in self._apply_bpe(match.group()))
        return ids
    
    def encode_iterable(self, iterable):
        """
        Given an iterable of strings (e.g. a Python file handle), return a generator that lazily yields token IDs. 
        This is required for memory-efficient tokenization of large files that we cannot directly load into memory.
        Args:
            iterable: Iterable[str]. Iterable of strings to encode.
        Returns:
            ids: Iterator[int]. Generator of iterable token IDs.
        """
        for text in iterable:
            yield from self.encode(text)
    
    def decode(self, ids):
        """
        Decode a sequence of token IDs into text.
        Args:
            ids: list[int]. Token IDs to decode into a string.
        Returns:
            text: str. String of decoded token IDs.
        """
        text = b''.join(self.vocab[id] for id in ids)
        return text.decode('utf-8', errors='replace')
    
    def _merge_pairs(self, ids, pair, new_id):
        new_ids = []
        i = 0
        while i < len(ids):
            if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
                new_ids.append(new_id)
                i += 2
            else:
                new_ids.append(ids[i])
                i += 1
        return new_ids

    def _apply_bpe(self, pretoken):
        tokens = [bytes([b]) for b in pretoken.encode('utf-8')]
        while len(tokens) >= 2:
            best_rank, best_idx = float('inf'), -1
            for i in range(len(tokens) - 1):
                rank = self.merge_ranks.get((tokens[i], tokens[i+1]), float('inf'))
                if rank < best_rank:
                    best_rank, best_idx = rank, i
            if best_idx == -1:
                break
            a, b = tokens[best_idx], tokens[best_idx + 1]
            tokens = self._merge_pairs(tokens, (a, b), a + b)
        return tokens

In [56]:
def get_tokenizer(vocab, merges, special_tokens):
    """Adapter function for the Tokenizer class."""
    return Tokenizer(vocab, merges, special_tokens=special_tokens)

test_roundtrip_empty(get_tokenizer)
test_empty_matches_tiktoken(get_tokenizer)
test_roundtrip_single_character(get_tokenizer)
test_single_character_matches_tiktoken(get_tokenizer)
test_roundtrip_single_unicode_character(get_tokenizer)
test_single_unicode_character_matches_tiktoken(get_tokenizer)
test_roundtrip_ascii_string(get_tokenizer)
test_ascii_string_matches_tiktoken(get_tokenizer)
test_roundtrip_unicode_string(get_tokenizer)
test_unicode_string_matches_tiktoken(get_tokenizer)
test_roundtrip_unicode_string_with_special_tokens(get_tokenizer)
test_unicode_string_with_special_tokens_matches_tiktoken(get_tokenizer)
test_overlapping_special_tokens(get_tokenizer)
test_address_roundtrip(get_tokenizer)
test_address_matches_tiktoken(get_tokenizer)
test_german_roundtrip(get_tokenizer)
test_german_matches_tiktoken(get_tokenizer)
test_tinystories_sample_roundtrip(get_tokenizer)
test_tinystories_matches_tiktoken(get_tokenizer)
test_encode_special_token_trailing_newlines(get_tokenizer)
test_encode_special_token_double_newline_non_whitespace(get_tokenizer)
test_encode_iterable_tinystories_sample_roundtrip(get_tokenizer)
test_encode_iterable_tinystories_matches_tiktoken(get_tokenizer)

PASSED: test_roundtrip_empty
PASSED: test_empty_matches_tiktoken
PASSED: test_roundtrip_single_character
PASSED: test_single_character_matches_tiktoken
PASSED: test_roundtrip_single_unicode_character
PASSED: test_single_unicode_character_matches_tiktoken
PASSED: test_roundtrip_ascii_string
PASSED: test_ascii_string_matches_tiktoken
PASSED: test_roundtrip_unicode_string
PASSED: test_unicode_string_matches_tiktoken
PASSED: test_roundtrip_unicode_string_with_special_tokens
PASSED: test_unicode_string_with_special_tokens_matches_tiktoken
PASSED: test_overlapping_special_tokens
PASSED: test_address_roundtrip
PASSED: test_address_matches_tiktoken
PASSED: test_german_roundtrip
PASSED: test_german_matches_tiktoken
PASSED: test_tinystories_sample_roundtrip
PASSED: test_tinystories_matches_tiktoken
PASSED: test_encode_special_token_trailing_newlines
PASSED: test_encode_special_token_double_newline_non_whitespace
PASSED: test_encode_iterable_tinystories_sample_roundtrip
PASSED: test_encode_iterab

## Experiments

Sample 10 documents from TinyStories and OpenWebText. Using your previously-trained TinyStories and OpenWebText tokenizers (10K and 32K vocabulary size, respectively), encode these sampled documents into integer IDs. What is each tokenizer's compression ratio (bytes/token)?

In [57]:
def sample_corpus(path, n_samples, doc_delimiter='<|endoftext|>', seed=seed):
    """Randomly samples a large corpus using Reservoir Sampling."""
    rng = np.random.default_rng(seed)
    samples = []
    parts = []
    seen = 0
    with open(path, 'r', encoding='utf-8', errors='replace') as file:
        for line in file:
            start = 0
            while True:
                i = line.find(doc_delimiter, start)
                if i < 0:
                    parts.append(line[start:])
                    break
                parts.append(line[start: i])
                doc = ''.join(parts)
                parts = []
                seen += 1
                if len(samples) < n_samples:
                    samples.append(doc)
                else:
                    j = rng.integers(seen)
                    if j < n_samples:
                        samples[j] = doc
                start = i + len(doc_delimiter)
    return samples

docs = sample_corpus(ts_valid_path, n_samples=10)
len(docs), docs[0]

(10,
 '\n\n\nBen and Mia were twins who liked to play with their toys. They had many toys, but their favorite was a big, hairy bear that they called Bobo. Bobo was soft and cuddly, and he had a red bow on his neck. Ben and Mia took Bobo everywhere, and they shared him at bedtime.\nOne day, Ben and Mia went to the park with their mom and dad. They brought Bobo with them, and they had fun on the swings, the slide, and the sandbox. But when they were ready to go home, they realized that Bobo was missing. They looked everywhere, but they could not find him. They felt very sad and cried.\n"Where is Bobo?" Ben asked.\n"I don\'t know, maybe someone took him," Mia said.\n"Maybe he ran away," Ben said.\n"No, he wouldn\'t do that, he loves us," Mia said.\nThey asked their mom and dad to help them look for Bobo, but they could not find him either. They had to leave the park without him.\nThat night, Ben and Mia could not sleep. They missed Bobo so much. They hugged each other and wished that Bobo

In [58]:
def tokenizer_compression_ratio(text, tokenizer):
    n_bytes = len(text.encode('utf-8'))
    n_tokens = len(tokenizer.encode(text))
    return n_bytes / n_tokens

tokenizer = Tokenizer.from_files(vocab_path=ts_vocab_path, merges_path=ts_merges_path, special_tokens=['<|endoftext|>'])
text = ''.join(docs)
tokenizer_compression_ratio(text, tokenizer)

4.015587114651888

In [59]:
ts_tokenizer = Tokenizer.from_files(vocab_path=ts_vocab_path, merges_path=ts_merges_path, special_tokens=['<|endoftext|>'])
ts_sample = sample_corpus(ts_train_path, n_samples=10)
tokenizer_compression_ratio(''.join(ts_sample), ts_tokenizer)

4.0

In [60]:
owt_tokenizer = Tokenizer.from_files(vocab_path=owt_vocab_path, merges_path=owt_merges_path, special_tokens=['<|endoftext|>'])
owt_sample = sample_corpus(owt_train_path, n_samples=10)
tokenizer_compression_ratio(''.join(owt_sample), owt_tokenizer)

4.578147268408551

What happens if you tokenize your OpenWebText sample with the TinyStories tokenizer? Compare the compression ratio and/or qualitatively describe what happens.

In [61]:
tokenizer_compression_ratio(''.join(owt_sample), ts_tokenizer)

3.297801351698178

Estimate the throughput of your tokenizer (e.g. in bytes/second). How long would it take to tokenize the *Pile* dataset (825GB of text)?

In [62]:
sample = sample_corpus(ts_train_path, n_samples=100)
text = ''.join(sample)
n_bytes = len(text.encode('utf-8'))
times = []
for run in range(10):
    start = time.time()
    ts_tokenizer.encode(text)
    times.append(time.time() - start)
avg_time = np.mean(times)
std_time = np.std(times)
throughput = n_bytes / avg_time
print(f'Time: {avg_time:.3f} +/- {std_time:.3f} sec')
print(f'Throughput: {throughput / 1e6:3f} MB/sec')

Time: 0.093 +/- 0.000 sec
Throughput: 0.887054 MB/sec


In [63]:
ts_size = 23e8  # 2.3 GB
ts_time = ts_size / throughput
print(f'TinyStories tokenization time: {ts_time / 60:3f} mins')

TinyStories tokenization time: 43.214189 mins


In [64]:
sample = sample_corpus(owt_train_path, n_samples=25)
text = ''.join(sample)
n_bytes = len(text.encode('utf-8'))
times = []
for run in range(10):
    start = time.time()
    owt_tokenizer.encode(text)
    times.append(time.time() - start)
avg_time = np.mean(times)
std_time = np.std(times)
throughput = n_bytes / avg_time
print(f'Time: {avg_time:.3f} +/- {std_time:.3f} sec')
print(f'Throughput: {throughput / 1e6:3f} MB/sec')

Time: 0.268 +/- 0.002 sec
Throughput: 0.716652 MB/sec


In [65]:
owt_size = 119e8  # 11.9 GB
owt_time = owt_size / throughput
print(f'OpenWebText tokenization time: {owt_time / 3600:3f} hours')

OpenWebText tokenization time: 4.612500 hours


In [66]:
pile_size = 825e9  # 825 GB
pile_time = pile_size / throughput
print(f'Pile tokenization time: {pile_time / (3600 * 24):3f} days')

Pile tokenization time: 13.323924 days


Our final task will be to use our implemented tokenizer to tokenize the TinyStories and OpenWebText datasets by encoding each corpus as a set of token IDs and serializing them to disk for later use, so we won't have to re-tokenize these datasets every time we train on them.

Implement the code below to tokenize the TinyStories *validation set* and serialize it to disk.  Once you've done that, do the exact same thing for the OpenWebText *validation set*. For each serialization, it's recommend to serialize each list of token IDs as a `np.array` of datatype `uint16`. Why is `uint16` a good choice here?

*Warning:* Do *not* use your serialization code on the *training sets* of either dataset. Doing so will likely crash your system! We'll explain why below and provide a more efficient workaround.

In [67]:
ts_valid_tokens_path = tmp_dir / 'ts-valid-tokens.bin'
with open(ts_valid_path, 'r', encoding='utf-8', errors='replace') as file:
    tokens = ts_tokenizer.encode_iterable(file)
    tokens = np.array(list(tokens), dtype=np.uint16)
    tokens.tofile(ts_valid_tokens_path)

In [68]:
owt_valid_tokens_path = tmp_dir / 'owt-valid-tokens.bin'
with open(owt_valid_path, 'r', encoding='utf-8', errors='replace') as file:
    tokens = owt_tokenizer.encode_iterable(file)
    tokens = np.array(list(tokens), dtype=np.uint16)
    tokens.tofile(owt_valid_tokens_path)

Now that we've tokenized and serialized the TinyStories and OpenWebText validation sets we're effectively done with these raw datasets for the rest of the course. From now on we can use the token IDs directly.

But we still have to deal with the much larger training datasets. Unfortunately, if you try to tokenize these larger datasets using your approach above it'll both take forever, and likely crash your system unless you're working on a machine with at least 100GB of memory. Instead it's recommended to use the helper function `tokenize_dataset_fast`, which has already been imported into this script. This helper function runs tokenization and serialization in parallel, and streams the output tokens to memory dynamically rather than all at once. This approach substantially reduces the memory footprint to avoid the kernel crashing, while also reducing the tokenization time by 50-90% depending on your CPU. To use this helper function, for each training set run something like this:

```python
tokenize_dataset_fast(input_path=text_file_path, output_path=output_tokens_path, vocab_path=vocab_path, merges_path=merges_path)
```

In [69]:
ts_train_tokens_path = tmp_dir / 'ts-train-tokens.bin'
tokenize_dataset_fast(input_path=ts_train_path, output_path=ts_train_tokens_path, vocab_path=ts_vocab_path, merges_path=ts_merges_path) # 10 mins

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2.23G/2.23G [10:11<00:00, 3.64MB/s]


In [70]:
owt_train_tokens_path = tmp_dir / 'owt-train-tokens.bin'
tokenize_dataset_fast(input_path=owt_train_path, output_path=owt_train_tokens_path, vocab_path=owt_vocab_path, merges_path=owt_merges_path) # 65 mins

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11.9G/11.9G [1:05:00<00:00, 3.06MB/s]
